# M5 Sales Forecasting — Memory-Efficient Pipeline

**Strategy:** optimize → melt → merge → delete, in that order.  
Each DataFrame is downcasted right after loading, and originals are freed after every merge.

In [ ]:
import pandas as pd
import numpy as np
import gc

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)  # safe limit — None can freeze on 58M rows

## 1. Memory Optimizer

Downcasts numeric columns to the smallest type that fits, and converts low-cardinality string columns to `category`.  
This cuts memory by 60–70% before any merge happens.

In [ ]:
def reduce_mem(df, label=""):
    """Downcast numeric dtypes and categorize low-cardinality object columns."""
    before = df.memory_usage(deep=True).sum() / 1e6

    for col in df.columns:
        col_type = df[col].dtype

        if col_type == object:
            # Categorize if cardinality is small relative to row count
            n_unique = df[col].nunique()
            if n_unique / len(df) < 0.5:
                df[col] = df[col].astype('category')
        elif col_type != bool:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type).startswith('int'):
                if c_min >= 0:
                    if c_max < 255:       df[col] = df[col].astype(np.uint8)
                    elif c_max < 65535:   df[col] = df[col].astype(np.uint16)
                    elif c_max < 4294967295: df[col] = df[col].astype(np.uint32)
                else:
                    if c_min > -128 and c_max < 127:      df[col] = df[col].astype(np.int8)
                    elif c_min > -32768 and c_max < 32767: df[col] = df[col].astype(np.int16)
                    else:                                   df[col] = df[col].astype(np.int32)
            elif str(col_type).startswith('float'):
                df[col] = df[col].astype(np.float32)

    after = df.memory_usage(deep=True).sum() / 1e6
    print(f"{label}: {before:.1f} MB → {after:.1f} MB ({100*(before-after)/before:.0f}% reduction)")
    return df

## 2. Load & Optimize Sales Data

In [ ]:
df_sales = pd.read_csv("sales_train_validation.csv")
df_sales = reduce_mem(df_sales, "df_sales")
print(df_sales.shape)
df_sales.head(3)

## 3. Melt Wide → Long + Extract d_num

`d_num` (integer day index) is kept throughout — it's essential for lag features, rolling windows, and the train/validation split.

In [ ]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
sales_cols = [col for col in df_sales.columns if col.startswith("d_")]

df_sales_long = df_sales.melt(
    id_vars=id_cols,
    value_vars=sales_cols,
    var_name="d",
    value_name="sales"
)

# Extract integer day number — keep it, don't drop
df_sales_long["d_num"] = df_sales_long["d"].str.extract(r"d_(\d+)").astype(np.uint16)

# Free the wide-format original immediately
del df_sales
gc.collect()

df_sales_long = reduce_mem(df_sales_long, "df_sales_long")
print(df_sales_long.shape)
df_sales_long.head()

## 4. Load & Optimize Calendar, then Merge

In [ ]:
df_calendar = pd.read_csv("calendar.csv")

# Parse date as datetime here — not after the merge
df_calendar["date"] = pd.to_datetime(df_calendar["date"])

# Fill event NaNs before optimizing so the column becomes a clean category
for col in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
    df_calendar[col] = df_calendar[col].fillna("No_Event")

df_calendar = reduce_mem(df_calendar, "df_calendar")
df_calendar.tail(3)

In [ ]:
df_merged = df_sales_long.merge(df_calendar, on="d", how="left")

# Free both inputs immediately after merging
del df_sales_long, df_calendar
gc.collect()

print(df_merged.shape)
print(df_merged.dtypes)
df_merged.head(3)

## 5. Load & Optimize Sell Prices, then Merge

Previously commented out — the join key is `[store_id, item_id, wm_yr_wk]`.

In [ ]:
df_sell_prices = pd.read_csv("sell_prices.csv")
df_sell_prices = reduce_mem(df_sell_prices, "df_sell_prices")
df_sell_prices.head()

In [ ]:
df_final = df_merged.merge(
    df_sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# Free both inputs immediately
del df_merged, df_sell_prices
gc.collect()

print(df_final.shape)
df_final.head(3)

## 6. Quick Sanity Checks

In [ ]:
print("Null counts:")
print(df_final.isnull().sum()[df_final.isnull().sum() > 0])

print("\nDate range:", df_final["date"].min(), "→", df_final["date"].max())
print("d_num range:", df_final["d_num"].min(), "→", df_final["d_num"].max())
print("Stores:", df_final["store_id"].unique().tolist())

## 7. Lag Features

Lag sales at 7, 14, and 28 days per item-store pair.  
`d_num` is what makes this possible — random-split approaches would leak future data.

In [ ]:
# Sort first so shift() operates in time order
df_final = df_final.sort_values(["id", "d_num"]).reset_index(drop=True)

for lag in [7, 14, 28]:
    df_final[f"sales_lag_{lag}"] = (
        df_final.groupby("id")["sales"].shift(lag).astype(np.float32)
    )

df_final[["id", "d_num", "sales", "sales_lag_7", "sales_lag_14", "sales_lag_28"]].head(10)

## 8. Time-Based Train / Validation Split

Split at day 1885 — the last 28 days (1886–1913) serve as the validation window.  
**Never use a random split on time-series data** — it lets the model see future values during training.

In [ ]:
SPLIT_DAY = 1885

df_train = df_final[df_final["d_num"] <= SPLIT_DAY].copy()
df_val   = df_final[df_final["d_num"] >  SPLIT_DAY].copy()

print(f"Train: {df_train.shape}  days 1–{SPLIT_DAY}")
print(f"Val:   {df_val.shape}  days {SPLIT_DAY+1}–{df_final['d_num'].max()}")

## 9. Save to Parquet

Parquet is ~10× smaller than CSV for this dataset and preserves `category` dtypes — no need to re-optimize on reload.

In [ ]:
df_train.to_parquet("train.parquet", index=False)
df_val.to_parquet("val.parquet", index=False)

print("Saved train.parquet and val.parquet")

# Reload check
pd.read_parquet("train.parquet").head(2)